## data cleaning

notebook used to generate `.txt` files in `data/`

In [14]:
# filter out puzzles with walls
with open("data/rush.txt", "r", encoding="utf-8") as fin, open("data/rush_no_walls.txt", "w", encoding="utf-8") as fout:
    for line in fin:
        stripped = line.strip()
        if not stripped:
            continue

        parts = stripped.split(maxsplit=2)
        if len(parts) != 3:
            continue

        _, board_desc, _ = parts
        if "x" not in board_desc:
            fout.write(line)

In [15]:
# get 4x4 puzzles
import pandas as pd

df = pd.read_json("hf://datasets/mustafaah/rushhour4x4-eval/dataset.json")

In [23]:
import re
import json

pattern = re.compile(r'Current Grid State \(JSON format\):\s*(\[\[.*?\]\])', re.DOTALL)

puzzles_4 = []
for _, row in df.iterrows():
    prompt = row["prompt"]
    m = pattern.search(prompt)
    if not m:
        continue
    try:
        grid = json.loads(m.group(1))
        moves = row["optimal_moves"]
        puzzles_4.append({
            "grid": grid,
            "optimal_moves": moves,
        })
    except json.JSONDecodeError:
        continue

In [24]:
# format and load 4x4 puzzles into their own file
# keep optimal_moves and assign a unique letter to each non-empty piece token.
# this preserves repeated cells for 2x1 blockers like h1.
def grid_to_string(grid):
    token_to_letter = {"C": "A"}
    next_letter = ord("B")
    result = []

    for row in grid:
        for cell in row:
            if cell == ".":
                result.append("o")
                continue

            if cell not in token_to_letter:
                token_to_letter[cell] = chr(next_letter)
                next_letter += 1

            result.append(token_to_letter[cell])

    return "".join(result)

with open("data/rush_4.txt", "w", encoding="utf-8") as f:
    for puzzle in puzzles_4:
        board = grid_to_string(puzzle["grid"])
        optimal_moves = puzzle["optimal_moves"]
        f.write(f"{optimal_moves} {board}\n")